# Data Import
Load JSON Lines files into Polars DataFrames for analysis.

In [26]:
# Imports
import json
import polars as pl
from pathlib import Path
from tqdm import tqdm

print("Imports OK")

Imports OK


In [27]:
# Configuration
DATA_DIR = Path("../data")
MODEL_FILE = DATA_DIR / "data-model-snapshop.json"
PREDICTOR_FILE = DATA_DIR / "data-Predictor-binning-snapshot.json"

def load_jsonl(filepath: Path, description: str = "Loading") -> list[dict]:
    """Load a JSON Lines file into a list of dictionaries.
    
    Args:
        filepath: Path to the .jsonl file
        description: Description for progress bar
    
    Returns:
        List of JSON objects
    """
    with open(filepath, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for _ in f)
    
    records = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in tqdm(f, total=total_lines, desc=description):
            records.append(json.loads(line))
    
    return records

# Load Model Snapshots
print("Loading model snapshots...")
model_records = load_jsonl(MODEL_FILE, "Model snapshots")
df_model_snapshots = pl.DataFrame(model_records)
print(f"  → {len(df_model_snapshots)} records, {len(df_model_snapshots.columns)} columns")

# Load Predictor Binning Snapshots
print("Loading predictor binning snapshots...")
predictor_records = load_jsonl(PREDICTOR_FILE, "Predictor binning")
df_predictor_binning = pl.DataFrame(predictor_records)
print(f"  → {len(df_predictor_binning)} records, {len(df_predictor_binning.columns)} columns")

print("\n✓ Data loaded successfully")

Loading model snapshots...


Model snapshots: 100%|██████████| 161704/161704 [00:01<00:00, 114221.37it/s]


  → 161704 records, 27 columns
Loading predictor binning snapshots...


Predictor binning: 100%|██████████| 222348/222348 [00:03<00:00, 64103.84it/s]


  → 222348 records, 34 columns

✓ Data loaded successfully


# Data Inspection
Explore the structure and content of both datasets.

In [28]:
# --- Model Snapshots ---
# One record per model (or model version)
print("=" * 60)
print("MODEL SNAPSHOTS")
print("=" * 60)

print(f"\nShape: {df_model_snapshots.shape}")
print(f"\nColumns:\n{df_model_snapshots.columns}")

print("\n--- Sample record ---")
df_model_snapshots.head(1).to_dicts()[0]

MODEL SNAPSHOTS

Shape: (161704, 27)

Columns:
['pyAppliesToClass', 'pyModelID', 'pyTotalPredictors', 'pxApplication', 'pyPerformance', 'pyPerformanceError', 'pyResponseCount', 'pyConfigurationName', 'pyFactoryUpdateTime', 'pyChannel', 'pyName', 'pyTreatment', 'pyDirection', 'pySnapshotTime', 'pxInsName', 'pyModelTechnique', 'pxSaveDateTime', 'pySuccessRate', 'pyGroup', 'pyNegatives', 'pyActivePredictors', 'pyPositives', 'pzInsKey', 'pyModelVersion', 'pxCommitDateTime', 'pyModelData', 'pyIssue']

--- Sample record ---


{'pyAppliesToClass': 'Data-Decision-Request-Customer',
 'pyModelID': '00116756-b49c-5a7a-a8cf-b503599fd40c',
 'pyTotalPredictors': 60,
 'pxApplication': 'PegaRULES',
 'pyPerformance': '0.500000',
 'pyPerformanceError': 0.0,
 'pyResponseCount': 113827.0,
 'pyConfigurationName': 'OmniAdaptiveModel',
 'pyFactoryUpdateTime': '20250909T100852.001 GMT',
 'pyChannel': 'Mobile',
 'pyName': 'DIVE',
 'pyTreatment': '',
 'pyDirection': 'Inbound',
 'pySnapshotTime': '20251017T050000.600 GMT',
 'pxInsName': '00116756-B49C-5A7A-A8CF-B503599FD40C!20251017T050000.600 GMT!PEGARULES',
 'pyModelTechnique': 'NaiveBayes',
 'pxSaveDateTime': '20251017T050056.206 GMT',
 'pySuccessRate': 0.0,
 'pyGroup': 'FlightAncillaries',
 'pyNegatives': 113827.0,
 'pyActivePredictors': 0,
 'pyPositives': 0.0,
 'pzInsKey': 'DATA-DECISION-ADM-MODELSNAPSHOT 00116756-B49C-5A7A-A8CF-B503599FD40C!20251017T050000.600 GMT!PEGARULES',
 'pyModelVersion': '635fbb86-0c06-5440-96b3-292c9abe6d98',
 'pxCommitDateTime': '20251017T050056.

In [29]:
# --- Predictor Binning Snapshots ---
# Multiple records per predictor (one per bin)
print("=" * 60)
print("PREDICTOR BINNING SNAPSHOTS")
print("=" * 60)

print(f"\nShape: {df_predictor_binning.shape}")
print(f"\nColumns:\n{df_predictor_binning.columns}")

# Unique predictors
unique_predictors = df_predictor_binning["pyPredictorName"].unique()
print(f"\n--- Predictor Summary ---")
print(f"Unique predictors: {len(unique_predictors)}")
print(f"Sample predictors: {unique_predictors[:5]}")

# Bins per predictor
bins_per_predictor = (
    df_predictor_binning
    .group_by("pyPredictorName")
    .agg(pl.len().alias("num_bins"))
    .sort("num_bins", descending=True)
)
print(f"\n--- Bins per Predictor (top 10) ---")
bins_per_predictor.head(10)

PREDICTOR BINNING SNAPSHOTS

Shape: (222348, 34)

Columns:
['pyBinResponseCountPercentage', 'pyContents', 'pyZRatio', 'pyModelID', 'pyBinType', 'pyBinSymbol', 'pyPredictorName', 'pyBinPositivesPercentage', 'pyBinNegativesPercentage', 'pyBinNegatives', 'pyType', 'pyBinLowerBound', 'pyPerformance', 'pyPerformanceError', 'pyResponseCount', 'pxCreateDateTime', 'pyBinIndex', 'pyLift', 'pyBinUpperBound', 'pyFeatureImportance', 'pySnapshotTime', 'pxInsName', 'pyTotalBins', 'pxSaveDateTime', 'pyNegatives', 'pyPositives', 'pyGroupIndex', 'pyEntryType', 'pzInsKey', 'pyBinPositives', 'pyBinResponseCount', 'pxCommitDateTime', 'pyRuleSetName', 'pxUpdateDateTime']

--- Predictor Summary ---
Unique predictors: 606
Sample predictors: shape: (5,)
Series: 'pyPredictorName' [str]
[
	"Customer.PreviousFlight.IsPost…
	"CustBookedFlight.Flight.IsFlig…
	"Event.EventType"
	"IH.Email.Inbound.Bounced.pxLas…
	"Customer.CustomerKPIs.DIVECoun…
]

--- Bins per Predictor (top 10) ---


pyPredictorName,num_bins
str,u32
"""Classifier""",12793
"""IH.Event.Outbound.RealTimeEven…",3129
"""IH.Email.Inbound.Delivered.pxL…",2984
"""IH.Email.Inbound.Pending.pxLas…",2956
"""IH.Email.Outbound.Pending.pxLa…",2918
"""IH.Email.Outbound.Delivered.px…",2870
"""IH.Email.Outbound.Clicked.pxLa…",2851
"""IH.Email.Outbound.Pending.pyHi…",2591
"""IH.Email.Inbound.Rejected.pxLa…",2477


In [31]:
# --- Key differences between datasets ---
print("=" * 60)
print("DATASET RELATIONSHIP")
print("=" * 60)

# Model ID links both datasets
model_ids_model = df_model_snapshots["pyModelID"].unique().to_list() if "pyModelID" in df_model_snapshots.columns else []
model_ids_predictor = df_predictor_binning["pyModelID"].unique().to_list()

print(f"\nModel IDs in model snapshots: {len(model_ids_model)}")
print(f"Model IDs in predictor binning: {len(model_ids_predictor)}")

if model_ids_model:
    common = set(model_ids_model) & set(model_ids_predictor)
    print(f"Common model IDs: {len(common)}")

DATASET RELATIONSHIP

Model IDs in model snapshots: 963
Model IDs in predictor binning: 963
Common model IDs: 963
